<p style="float: left;"><a href="generic-classes.ipynb" target="_blank">Previous</a></p>
<p style="float: right;"><a href="upper-type-bounds.ipynb" target="_blank">Next</a></p>
<p style="text-align:center;">Tour of Scala</p>
<div style="clear: both;"></div>

# Variances

Variance lets you control how type parameters behave with regard to subtyping. Scala supports variance annotations of type parameters of [generic classes](generic-classes.ipynb), to allow them to be covariant, contravariant, or invariant if no annotations are used. The use of variance in the type system allows us to make intuitive connections between complex types.

In [1]:
class Foo[+A] // A covariant class
class Bar[-A] // A contravariant class
class Baz[A]  // An invariant class

defined class Foo
defined class Bar
defined class Baz

### Invariance

By default, type parameters in Scala are invariant: subtyping relationships between the type parameters aren't reflected in the parameterized type. To explore why this works the way it does, we look at a simple parameterized type, the mutable box.

In [2]:
class Box[A](var content: A)

defined class Box

We're going to be putting values of type `Animal` in it. This type is defined as follows:

In [3]:
abstract class Animal {
  def name: String
}
case class Cat(name: String) extends Animal
case class Dog(name: String) extends Animal

defined class Animal
defined class Cat
defined class Dog

We can say that `Cat` is a subtype of `Animal`, and that `Dog` is also a subtype of `Animal`. That means that the following is well-typed:

In [4]:
val myAnimal: Animal = Cat("Felix")

myAnimal: Animal = Cat(name = "Felix")

What about boxes? Is `Box[Cat]` a subtype of `Box[Animal]`, like `Cat` is a subtype of `Animal`? At first sight, it looks like that may be plausible, but if we try to do that, the compiler will tell us we have an error:

In [5]:
val myCatBox: Box[Cat] = new Box[Cat](Cat("Felix"))
val myAnimalBox: Box[Animal] = myCatBox // this doesn't compile
val myAnimal: Animal = myAnimalBox.content

cmd5.sc:2: type mismatch;
 found   : cmd5.this.cmd2.Box[cmd5.this.cmd3.Cat]
 required: cmd5.this.cmd2.Box[cmd5.this.cmd3.Animal]
Note: cmd5.this.cmd3.Cat <: cmd5.this.cmd3.Animal, but class Box is invariant in type A.
You may wish to define A as +A instead. (SLS 4.5)
val myAnimalBox: Box[Animal] = myCatBox // this doesn't compile
                               ^
Compilation Failed

: Compilation Failed

Why could this be a problem? We can get the cat from the box, and it's still an Animal, isn't it? Well, yes. But that's not all we can do. We can also replace the cat in the box with a different animal

```scala
  myAnimalBox.content = Dog("Fido")
```

There now is a Dog in the Animal box. That's all fine, you can put Dogs in Animal boxes, because Dogs are Animals. But our Animal Box is a Cat Box! You can't put a Dog in a Cat box. If we could, and then try to get the cat from our Cat Box, it would turn out to be a dog, breaking type soundness.

```scala
  val myCat: Cat = myCatBox.content //myCat would be Fido the dog!
```

From this, we have to conclude that `Box[Cat]` and `Box[Animal]` can't have a subtyping relationship, even though `Cat` and `Animal` do.

### Covariance

The problem we ran in to above, is that because we could put a Dog in an Animal Box, a Cat Box can't be an Animal Box.

But what if we couldn't put a Dog in the box? Then, we could just get our Cat back out without a problem, and it would adhere to the subtyping relationship. It turns out that that's possible to do.

In [5]:
class ImmutableBox[+A](val content: A)
val catbox: ImmutableBox[Cat] = new ImmutableBox[Cat](Cat("Felix"))
val animalBox: ImmutableBox[Animal] = catbox // now this compiles

defined class ImmutableBox
catbox: ImmutableBox[Cat] = ammonite.$sess.cmd5$Helper$ImmutableBox@68afe132
animalBox: ImmutableBox[Animal] = ammonite.$sess.cmd5$Helper$ImmutableBox@68afe132

We say that `ImmutableBox` is *covariant* in `A`, and this is indicated by the `+` before the `A`.

More formally, that gives us the following relationship: given some `class Cov[+T]`, then if `A` is a subtype of `B`, `Cov[A]` is a subtype of `Cov[B]`. This allows us to make very useful and intuitive subtyping relationships using generics.

In the following less contrived example, the method `printAnimalNames` will accept a list of animals as an argument and print their names each on a new line. If `List[A]` were not covariant, the last two method calls would not compile, which would severely limit the usefulness of the `printAnimalNames` method.

In [6]:
def printAnimalNames(animals: List[Animal]): Unit =
  animals.foreach {
    animal => println(animal.name)
  }

val cats: List[Cat] = List(Cat("Whiskers"), Cat("Tom"))
val dogs: List[Dog] = List(Dog("Fido"), Dog("Rex"))

// prints: Whiskers, Tom
printAnimalNames(cats)

// prints: Fido, Rex
printAnimalNames(dogs)

Whiskers
Tom
Fido
Rex


defined function printAnimalNames
cats: List[Cat] = List(Cat(name = "Whiskers"), Cat(name = "Tom"))
dogs: List[Dog] = List(Dog(name = "Fido"), Dog(name = "Rex"))

### Contravariance

We've seen we can accomplish covariance by making sure that we can't put something in the covariant type, but only get something out. What if we had the opposite, something you can put something in, but can't take out? This situation arises if we have something like a serializer, that takes values of type A, and converts them to a serialized format.

In [7]:
abstract class Serializer[-A] {
  def serialize(a: A): String
}

val animalSerializer: Serializer[Animal] = new Serializer[Animal] {
  def serialize(animal: Animal): String = s"""{ "name": "${animal.name}" }"""
}
val catSerializer: Serializer[Cat] = animalSerializer
catSerializer.serialize(Cat("Felix"))

defined class Serializer
animalSerializer: Serializer[Animal] = ammonite.$sess.cmd7$Helper$$anon$1@1f626db8
catSerializer: Serializer[Cat] = ammonite.$sess.cmd7$Helper$$anon$1@1f626db8
res7_3: String = "{ \"name\": \"Felix\" }"

We say that `Serializer` is *contravariant* in `A`, and this is indicated by the `-` before the `A`. A more general serializer is a subtype of a more specific serializer.

More formally, that gives us the reverse relationship: given some `class Contra[-T]`, then if `A` is a subtype of `B`, `Contra[B]` is a subtype of `Contra[A]`.

### Immutability and Variance
Immutability constitutes an important part of the design decision behind using variance. For example, Scala's collections systematically distinguish between [mutable and immutable collections](https://docs.scala-lang.org/overviews/collections-2.13/overview.html). The main issue is that a covariant mutable collection can break type safety. This is why `List` is a covariant collection, while `scala.collection.mutable.ListBuffer` is an invariant collection. `List` is a collection in package `scala.collection.immutable`, therefore it is guaranteed to be immutable for everyone. Whereas, `ListBuffer` is mutable, that is, you can change, add, or remove elements of a `ListBuffer`.

To illustrate the problem of covariance and mutability, suppose that `ListBuffer` was covariant, then the following problematic example would compile (in reality it fails to compile):

In [8]:
import scala.collection.mutable.ListBuffer

val bufInt: ListBuffer[Int] = ListBuffer[Int](1,2,3)
val bufAny: ListBuffer[Any] = bufInt
bufAny(0) = "Hello"
val firstElem: Int = bufInt(0)

cmd8.sc:4: type mismatch;
 found   : scala.collection.mutable.ListBuffer[Int]
 required: scala.collection.mutable.ListBuffer[Any]
Note: Int <: Any, but class ListBuffer is invariant in type A.
You may wish to investigate a wildcard type such as `_ <: Any`. (SLS 3.2.10)
val bufAny: ListBuffer[Any] = bufInt
                              ^
Compilation Failed

: Compilation Failed

If the above code was possible then evaluating `firstElem` would fail with `ClassCastException`, because `bufInt(0)` now contains a `String`, not an `Int`.

The invariance of `ListBuffer` means that `ListBuffer[Int]` is not a subtype of `ListBuffer[Any]`, despite the fact that `Int` is a subtype of `Any`, and so `bufInt` cannot be assigned as the value of `bufAny`.

### Comparison With Other Languages

Variance is supported in different ways by some languages that are similar to Scala. For example, variance annotations in Scala closely resemble those in C#, where the annotations are added when a class abstraction is defined (declaration-site variance). In Java, however, variance annotations are given by clients when a class abstraction is used (use-site variance).

Scala's tendency towards immutable types makes it that covariant and contravariant types are more common than in other languages, since a mutable generic type must be invariant.
<p style="float: left;"><a href="generic-classes.ipynb" target="_blank">Previous</a></p>
<p style="float: right;"><a href="upper-type-bounds.ipynb" target="_blank">Next</a></p>
<p style="text-align:center;">Tour of Scala</p>
<div style="clear: both;"></div>